# Suspended IDs

This notebook grabs additional data to flesh out suspect user IDs. PRAW is not available for suspended accounts, meaning we need to source data elsewhere. This process is different than the active account info collection because you can use Reddit.

In [ ]:
# ARCHIVED draft of the Arctic Shift post-history pull (replaced by p2_data_cleaning/collection_scripts/tier4_arctic_full_history.py).
# Libraries.
#import libraries
import pandas as pd
import numpy as np
import os 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
from string import punctuation
import functools
import operator
from dotenv import load_dotenv
import praw
import prawcore
from prawcore.exceptions import NotFound
import pprint
import ast
import pprint
import os 
import itertools
import pickle
import requests
from time import time
from requests_ratelimiter import LimiterSession
import pprint
import json
import base64
from itertools import islice
import pyarrow as pa

In [ ]:
# Accounts from the first collection (User_Profiles.parquet).
users = pd.read_parquet('User_Profiles.parquet', engine='pyarrow')

In [ ]:
# First 5 accounts as a test set.
u = users['username'][0:5]

In [ ]:
# Page through each account's posts (100 per request, 5 requests/second, up to 10 tries per page),
# then flatten to one row per post with the _meta fields as columns and outer_key = the account.
def get_historic_data(name_list):
    session = LimiterSession(per_second=5)
    user_submissions = {}
    
    for name in name_list:
        print(name)
        all_posts = []
        after = None
        
        while True:
            arctic_url = f'https://arctic-shift.photon-reddit.com/api/posts/search?author={name}&limit=100'
            if after:
                arctic_url += f'&after={after}'
            
            retries = 0
            success = False
            
            while retries < 10:
                print(f'Retries are at: {retries}')
                try:
                    response = session.get(arctic_url)
                    
                    if response.status_code == 200:
                        api_return = response.__dict__
                        data_content = json.loads(api_return
                                                  .get('_content')
                                                  .decode('utf-8'))
                        
                        posts = data_content['data']
                        all_posts.extend(posts)
                        
                        if len(posts) < 100:
                            success = True
                            break
                        
                        after = posts[-1]['created_utc']
                        success = True
                        break
                    
                    else:
                        retries += 1
                        print(f'API response: {response.status_code}')
                
                except Exception as e:
                    print(f"Hit unexpected error with class {e.__class__}:{e}")
                    raise e
            
            if not success or len(posts) < 100:
                break
        
        user_submissions[name] = all_posts
    
    # Build DataFrame from all users
    rows = []
    for outer_key, records in user_submissions.items():
        for record in records:
            meta = record.get('_meta', {})
            row = {'outer_key': outer_key} | meta | {k: v for k, v in record.items() if k != '_meta'}
            rows.append(row)

    df = pd.DataFrame(rows)
    
    return df

In [ ]:
# Test run on the 5 accounts.
test=get_historic_data(u)

In [ ]:
test.head(2)